# 2.3 Simpson's Paradox

**Is dit een visualisatieles of een statistiekles? Het is een statistiekles.** 

De visualisaties die we hier gebruiken zijn eenvoudig — gegroepeerde staven en een *barbell plot* (haltergrafiek). Het probleem dat ze zichtbaar maken is dat echter allerminst: een vergelijking die op totaalniveau een overtuigend effect laat zien, kan op subgroepniveau **volledig omdraaien**, zonder dat er ook maar één rekenfout is gemaakt.

Dit heet **Simpson's Paradox**. Als je dit eenmaal hebt gezien, kijk je nooit meer met dezelfde ogen naar een geaggregeerd gemiddelde.

In [ ]:
# Paragraaf 2.3: Benodigde bibliotheken en visualisatieklassen importeren
import pandas as pd
from goad_toolkit.visualizer import BarbellPlot, GroupedBarPlot, HighlightCategory, PlotSettings
from scripts.plots import BarPlot
from wa_analyzer.data import load_showcase

## 2.3.1 De situatie: Toelatingen tot Berkeley (1973)

Toelatingen tot de masteropleidingen van UC Berkeley in 1973 — een van de beroemdste en meest geciteerde praktijkvoorbeelden uit de statistiek. 

Zes grote faculteiten (departments A t/m F), het aantal aanmeldingen (`applied`) en het aantal toegelaten studenten (`admitted`), uitgesplitst naar man en vrouw.

In [ ]:
# Paragraaf 2.3.1: Berkeley-dataset inladen
berkeley = load_showcase("berkeley_admissions")
berkeley.head(6)  # Bekijk de data: faculteit (A t/m F), geslacht, aanmeldingen, toelatingen

## 2.3.2 Eerst de verkeerde conclusie trekken

Gooi alle faculteiten op één grote hoop en vergelijk simpelweg het totale toelatingspercentage tussen mannen en vrouwen:

In [ ]:
# Paragraaf 2.3.2: Geaggregeerde toelatingspercentages berekenen
overall = berkeley.groupby("gender")[["applied", "admitted"]].sum().reset_index()
overall["rate"] = overall.admitted / overall.applied * 100

print(overall)  # Mannen: 44.5% toegelaten, Vrouwen: 30.4% toegelaten

Op het eerste gezicht: **44,5% van de mannen werd toegelaten, tegenover slechts 30,4% van de vrouwen.** 

Een verschil van ruim 14 procentpunten! Bij een steekproef van duizenden studenten is dit statistisch verpletterend. De voor de hand liggende krantenkop schrijft zichzelf: *"Universiteit discrimineert massaal tegen vrouwelijke studenten"*.

**Voordat je dit gelooft, stel je jezelf altijd de ene vraag die je hiertegen beschermt:**
> Is er een variabele die verschilt tussen de groepen die we vergelijken, én die tegelijkertijd invloed heeft op de uitkomst?

Als zo'n variabele bestaat, meet het totaalcijfer niet wat je denkt dat het meet, totdat je binnen die variabele hebt gekeken. Hier zit die variabele gewoon recht voor onze neus in de data: **de faculteit (`department`)**.

## 2.3.3 De diepte in: splitsen per subgroep

Exact dezelfde vergelijking, maar nu per faculteit apart berekend. 

We zetten een `GroupedBarPlot` direct naast de totaalgrafiek, zodat je beide beelden naast elkaar kunt vergelijken:

In [ ]:
# Paragraaf 2.3.3: Totaal vergelijken met uitsplitsing per faculteit
berkeley["rate"] = berkeley.admitted / berkeley.applied * 100

split = PlotSettings(
    figsize=(13, 4),
    title="Berkeley 1973, two levels of aggregation",
    subplot_titles=["pooled: men admitted more often",
                    "per department: mostly the opposite"],
    xlabel="",
    ylabel="admission rate (%)",
)

host = BarPlot(split)
fig, axes = host.create_figure(n_plots=2)

# Links: Totaalbeeld (mannen lijken zwaar bevoordeeld)
host.plot_on_axes(BarPlot(split), axes[0], data=overall, x="gender", y="rate", color="#cccccc")

# Rechts: Per faculteit (vrouwen worden bij de meeste faculteiten juist vaker toegelaten!)
host.plot_on_axes(GroupedBarPlot(split), axes[1],
                  data=berkeley, x="department", y="rate", hue="gender")

Kijk goed wat er gebeurt:
* **Links (geaggregeerd):** Mannen worden veel vaker toegelaten.
* **Rechts (per faculteit):** Bij de meeste faculteiten worden vrouwen juist vaker toegelaten!

Niets hiervan is een rekenfout — beide grafieken zijn wiskundig 100% correcte weergaven van exact dezelfde dataset van twaalf rijen.

Het rechterpaneel toont de omkering al per faculteit — maar de vraag *"bij welke faculteiten en met welk verschil"* beantwoordt een tabel met getallen vaak net iets te abstract. 

Een **barbell plot** (halterdiagram) tekent het mannen/vrouwen-paar per faculteit als twee stippen verbonden door een lijn. Daardoor zie je in één oogopslag niet alleen de eindpunten, maar vooral **het verschil (the gap)** tussen beide groepen:

In [ ]:
# Paragraaf 2.3.3: Barbell Plot (haltergrafiek van het verschil per faculteit)
wide = berkeley.pivot(index="department", columns="gender", values="rate").reset_index()
wide["gap"] = wide.women - wide.men  # Positief betekent dat vrouwen vaker worden toegelaten
wide = wide.sort_values("gap")

barbell_settings = PlotSettings(
    figsize=(9, 4),
    title="Admission rate by department: men vs women",
    xlabel="admission rate (%)",
    ylabel="department",
)
fig, ax = BarbellPlot(barbell_settings).plot(
    data=wide, category="department", start="men", end="women",
    start_label="men", end_label="women",
)

favouring_women = int((wide.gap > 0).sum())
print(f"women admitted at a higher rate in {favouring_women} of {len(wide)} departments")

Bij vier van de zes faculteiten ligt het rode bolletje (vrouwen) rechts van het grijze bolletje (mannen). Slechts bij twee faculteiten is het andersom (en met een heel klein verschil). 

Dat is de hele paradox in één beeld gevangen: **op totaalniveau wint grijs (mannen); per faculteit wint rood (vrouwen).**

## 2.3.4 Waarom draait het effect om? (De verstorende variabele)

Het mechanisme zit in het **aanmeldgedrag**, niet in de toelatingsbeslissingen van de professoren!

Hieronder plotten we eerst hoe selectief elke faculteit is (het toelatingspercentage op totaalniveau — wat je in de geaggregeerde grafiek nooit zag), en daarnaast het percentage vrouwelijke aanmelders per faculteit, in exact dezelfde volgorde:

In [ ]:
# Paragraaf 2.3.4: Het mechanisme onthuld (selectiviteit vs aanmelders)
from matplotlib.patches import Patch

applied = berkeley.pivot(index="department", columns="gender", values="applied")
admitted = berkeley.pivot(index="department", columns="gender", values="admitted")

mechanism = pd.DataFrame({
    "department": applied.index,
    "overall_rate": (admitted.men + admitted.women) / (applied.men + applied.women) * 100,
    "women_share": applied.women / (applied.men + applied.women) * 100,
}).sort_values("overall_rate", ascending=False)

hardest_two = mechanism.nsmallest(2, "overall_rate").department.tolist()

mech_settings = PlotSettings(
    figsize=(13, 4),
    title="Selectivity predicts who applied where",
    subplot_titles=["overall admission rate, easiest to hardest department",
                    "% of applicants who are women, same order"],
    xlabel="department",
    highlight=hardest_two,
)

host = BarPlot(mech_settings)
fig, axes = host.create_figure(n_plots=2)

# Links: Hoe selectief is de faculteit? (van makkelijk naar zwaar)
host.plot_on_axes(BarPlot(mech_settings), axes[0], data=mechanism, x="department",
                  y="overall_rate", order=mechanism.department, color="#cccccc")
host.plot_on_axes(HighlightCategory(mech_settings), axes[0])
axes[0].set_ylabel("admission rate (%)")

# Rechts: Hoeveel procent van de aanmelders is vrouw?
host.plot_on_axes(BarPlot(mech_settings), axes[1], data=mechanism, x="department",
                  y="women_share", order=mechanism.department, color="#cccccc")
host.plot_on_axes(HighlightCategory(mech_settings), axes[1])
axes[1].set_ylabel("% of applicants who are women")

axes[0].legend(
    handles=[
        Patch(facecolor=mech_settings.base_color, label="other departments"),
        Patch(facecolor=mech_settings.highlight_color, label="the two hardest to get into"),
    ],
    loc="upper right", fontsize=8,
)

Leg de twee panelen naast elkaar en de oorzaak is direct glashelder:
* De twee zwaarste en meest selectieve faculteiten (A en B, in het rood geaccentueerd) laten bijna niemand toe (<10% toelating). En laat dát nou net de faculteiten zijn waar **een gigantisch percentage vrouwen** zich voor aanmeldde!
* Mannen meldden zich daarentegen massaal aan bij de makkelijkste faculteiten (links in de grafiek), waar meer dan 60% tot 80% van de aanmelders gewoon wordt aangenomen.

Het totale toelatingspercentage mat dus vooral **op welke faculteiten mensen solliciteerden**, en niet hoe individuele aanmelders door een toelatingscommissie werden behandeld. Binnen elke individuele faculteit werden vrouwen net zo vaak (of zelfs vaker) toegelaten dan mannen. 

Het geaggregeerde getal leek op discriminatie, simpelweg omdat studiekeuze en geslacht met elkaar verstrengeld waren in de aanmelderspopulatie.

## 2.3.5 Het stappenplan voor je eigen data

Deze bescherming geldt universeel voor elke data-analyse:

> **Elke keer dat een geaggregeerde vergelijking tussen twee groepen overtuigend lijkt, vraag jezelf af:**
> Is er een variabele die verschilt tussen de groepen én die de uitkomst beïnvloedt? Zo ja, splits je data op die variabele en vergelijk subgroep voor subgroep, vóórdat je het totaalcijfer gelooft.

Het moeilijke is nooit de `groupby` in Python. Het moeilijke is **opmerken dat er een verborgen verstorende variabele (*lurking variable*) bestaat**, nog vóórdat je de conclusie van je analyse hebt opgeschreven.

De gebruikelijke verdachten in chat- en communicatiedata:
1. **Tijdsperiode:** Een groep of platform verandert over de maanden heen; alles op één hoop gooien kan een trend verbergen of juist kunstmatig creëren.
2. **Activiteitsniveau of ervaring:** Nieuwe leden versus vaste stamgasten; toevallige voorbijgangers versus de acht diehards die de helft van alle berichten typen (hier gaat notebook 2.4 over!).
3. **Subkanaal of onderwerp:** Een vergelijking tussen twee groepsgesprekken kan gedreven worden door wáár het gesprek over gaat, in plaats van wie erin zit.
4. **Apparaat of besturingssysteem:** iPhone vs Android (verschillende timestamps, andere emoji-representaties).
5. **Doordeweeks vs Weekend, of tijdstip van de dag:** Een groep die toevallig vaker 's avonds praat vertoont ander gedrag dan een groep die overdag chat.

**Hoe controleer je dit?**
Splits de geaggregeerde vergelijking op de verdachte variabele (`groupby` op twee kolommen in plaats van één), plot de subgroepen via een `GroupedBarPlot` of `BarbellPlot`, en zoek naar een **omkering van het effect**, niet alleen naar een verandering in grootte. Blijft het effect in elke subgroep dezelfde kant op wijzen? Dan klopte je totaalcijfer. Draait het om? Dan heb je een veel interessanter inzicht ontdekt!

---

**Hoe we nu verder gaan:** Simpson's paradox ontstaat door een verborgen *groeperingsvariabele*. 

In [02.4-counting-and-averages](02.4-counting-and-averages.ipynb) zien we een variant die nóg subtieler is: eentje die ontstaat door een onzorgvuldige keuze van **wát je telt** — zonder dat er een derde variabele aan te pas komt!